In [0]:
%python
# dml/04_carga_stg_scoring_papel.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("🚀 Iniciando cálculo de Scoring, Métricas de Qualidade e Ranking Desempatado para FIIs de Papel (Recebíveis)...")

# %%
# 1. Busca e calcula as métricas reais cruzando Yahoo Finance com Staging CVM
# Aplicamos a trava de liquidez, desduplicação temporal e a regra de normalização de dividendos
qry_calculo_metricas = f"""
  WITH de_para_unico AS (
    -- Garante CNPJ e segmento únicos por ticker
    SELECT 
      ticker, 
      MAX(cnpj) as cnpj,
      MAX(segmento_atuacao) as segmento_atuacao
    FROM {catalogo}.{schema}.dim_fundo_imobiliario
    WHERE cnpj IS NOT NULL
    GROUP BY ticker
  ),

  complemento_12m_acumulado AS (
    -- Soma os Dividend Yields mensais oficiais da CVM dos últimos 12 meses (Livre de Amortizações!)
    -- E captura a foto cadastral mais recente de cotistas e PL
    SELECT 
      ticker,
      SUM(TRY_CAST(percentual_dividend_yield_mes AS DOUBLE)) AS soma_dy_patrimonial_12m,
      MAX(TRY_CAST(total_numero_cotistas AS INT)) AS total_cotistas,
      -- Pega a foto mais recente do valor patrimonial da cota para conversão
      FIRST_VALUE(TRY_CAST(valor_patrimonial_cota AS DOUBLE)) OVER (PARTITION BY ticker ORDER BY data_referencia DESC) AS valor_patrimonial_cota,
      FIRST_VALUE(TRY_CAST(patrimonio_liquido AS DOUBLE)) OVER (PARTITION BY ticker ORDER BY data_referencia DESC) AS patrimonio_liquido
    FROM {catalogo}.{schema}.stg_cvm_informe_complemento
    WHERE data_referencia >= ADD_MONTHS(CURRENT_DATE(), -12)
    GROUP BY ticker, data_referencia, valor_patrimonial_cota, patrimonio_liquido
  ),
  
  complemento_recente AS (
    -- Garante apenas 1 registro consolidado dos dados de complemento mais recentes
    SELECT * FROM (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY soma_dy_patrimonial_12m DESC) as rn
      FROM complemento_12m_acumulado
    ) WHERE rn = 1
  ),

  ativo_passivo_dedup AS (
    -- Garante apenas 1 registro (o mais recente) por ticker da CVM para ativos e passivos
    SELECT * FROM (
      SELECT 
        ticker,
        total_investido,
        valor_investido_cri_cra,
        valor_caixa_disponibilidades,
        total_passivo_obrigacoes,
        ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY data_referencia DESC, data_carga DESC) as rn
      FROM {catalogo}.{schema}.stg_cvm_informe_ativo_passivo
    ) WHERE rn = 1
  ),

  historico_recente AS (
    -- Avalia a liquidez recente dos últimos 30 dias
    SELECT 
      ticker,
      preco_fechamento,
      volume_negociado,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -1)
  ),
  
  liquidez_fiis AS (
    -- Filtra apenas fundos com liquidez real ativa
    SELECT 
      ticker,
      AVG(volume_negociado) AS volume_medio_diario,
      MAX(data_pregao) AS data_ultimo_negocio
    FROM historico_recente
    WHERE volume_negociado > 0
    GROUP BY ticker
    HAVING volume_medio_diario >= 50 AND data_ultimo_negocio >= DATE_SUB(CURRENT_DATE(), 15)
  ),
  
  precos_atuais AS (
    -- Captura o último preço de mercado ativo
    SELECT ticker, preco_atual
    FROM (
      SELECT ticker, preco_fechamento AS preco_atual, ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY data_pregao DESC) as rn
      FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
      WHERE volume_negociado > 0
    ) WHERE rn = 1
  ),
  
  cadastro_papel AS (
    -- Filtra apenas FIIs de Papel ativos no de-para
    SELECT ticker, segmento_atuacao
    FROM de_para_unico
    WHERE segmento_atuacao IN ('Papel (Recebíveis Imobiliários)', 'Títulos e Valores Mobiliários')
  )
  
  -- Junta mercado (Yahoo) com contabilidade oficial desduplicada (CVM)
  SELECT 
    c.ticker,
    c.segmento_atuacao,
    p.preco_atual,
    comp.valor_patrimonial_cota,
    comp.patrimonio_liquido,
    -- Captura o Ativo de CRIs real da CVM. Se nulo, faz fallback para o total investido
    COALESCE(ap.valor_investido_cri_cra, ap.total_investido) AS valor_portfolio_cri,
    
    -- CONVERSÃO DO YIELD DA CVM: (Soma DY Patrimonial 12M) * (Valor Patrimonial / Preço Atual) * 100
    -- Isso expurga 100% as amortizações das séries históricas de recebíveis!
    ROUND(
      COALESCE(
        try_divide(comp.soma_dy_patrimonial_12m * comp.valor_patrimonial_cota, p.preco_atual) * 100.0, 
        0.0
      ), 2
    ) AS dividend_yield_12m,
    
    -- Métricas de Qualidade e Desempate
    COALESCE(comp.total_cotistas, 0) AS total_cotistas,
    ROUND(COALESCE(try_divide(ap.total_passivo_obrigacoes, ap.total_investido) * 100.0, 0.0), 2) AS alavancagem_percentual,
    ROUND(COALESCE(try_divide(ap.valor_caixa_disponibilidades, ap.total_investido) * 100.0, 0.0), 2) AS caixa_percentual
  FROM cadastro_papel c
  INNER JOIN liquidez_fiis l ON c.ticker = l.ticker
  INNER JOIN precos_atuais p ON c.ticker = p.ticker
  INNER JOIN de_para_unico dp ON c.ticker = dp.ticker
  INNER JOIN complemento_recente comp ON c.ticker = comp.ticker
  INNER JOIN ativo_passivo_dedup ap ON c.ticker = ap.ticker
"""

df_metricas = spark.sql(qry_calculo_metricas)
df_metricas.createOrReplaceTempView("v_metricas_base_papel_real")

# %%
# 2. Aplicação do Scoring e Risco de Crédito Real para Papel (Blindado contra NaNs e zeros)
qry_scoring = f"""
  WITH limites AS (
    -- Tratamento defensivo definitivo contra NaNs e Nulos nos agregadores globais
    SELECT 
      COALESCE(MAX(nanvl(dividend_yield_12m, 0.0)), 0.0) as max_div,
      COALESCE(MIN(nanvl(dividend_yield_12m, 0.0)), 0.0) as min_div,
      COALESCE(MAX(nanvl(valor_portfolio_cri, 0.0)), 0.0) as max_cri,
      COALESCE(MIN(nanvl(valor_portfolio_cri, 0.0)), 0.0) as min_cri
    FROM v_metricas_base_papel_real
  ),
  
  scores_calculados AS (
    SELECT 
      m.ticker,
      m.preco_atual,
      m.valor_patrimonial_cota,
      ROUND(COALESCE(try_divide(m.preco_atual, m.valor_patrimonial_cota), 0.0), 2) AS p_vp,
      m.dividend_yield_12m,
      m.valor_portfolio_cri,
      m.patrimonio_liquido,
      
      -- Métricas de desempate herdadas para o SELECT final
      m.total_cotistas,
      m.alavancagem_percentual,
      m.caixa_percentual,
      
      -- Normalizações seguras utilizando try_divide e tratando NaNs de forma nativa no Spark
      ROUND(
        COALESCE(
          try_divide((nanvl(m.dividend_yield_12m, 0.0) - l.min_div), (l.max_div - l.min_div)) * 100.0, 
          0.0
        ), 2
      ) AS nota_dy,
      
      ROUND(
        COALESCE(
          try_divide((nanvl(m.valor_portfolio_cri, 0.0) - l.min_cri), (l.max_cri - l.min_cri)) * 100.0, 
          0.0
        ), 2
      ) AS nota_portfolio,
      
      -- Normalização para P/VP em Papel (Próximo de 0.99 é o ideal/nota 100)
      CASE 
        WHEN COALESCE(try_divide(m.preco_atual, m.valor_patrimonial_cota), 0.0) BETWEEN 0.96 AND 1.02 THEN 100.0
        WHEN COALESCE(try_divide(m.preco_atual, m.valor_patrimonial_cota), 0.0) < 0.96 
          THEN ROUND(GREATEST(0.0, (1.0 - (0.96 - COALESCE(try_divide(m.preco_atual, m.valor_patrimonial_cota), 0.0)) * 2.0) * 100.0), 2)
        ELSE ROUND(GREATEST(0.0, (1.0 - (COALESCE(try_divide(m.preco_atual, m.valor_patrimonial_cota), 0.0) - 1.02) * 5.0) * 100.0), 2)
      END AS nota_pvp
    FROM v_metricas_base_papel_real m
    CROSS JOIN limites l
  )
  
  SELECT 
    ticker,
    CURRENT_DATE() AS data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    valor_portfolio_cri,
    
    -- Métricas de Desempate
    total_cotistas,
    alavancagem_percentual,
    caixa_percentual,
    
    -- Média Ponderada: 40% P/VP, 40% DY, 20% Diversificação/Tamanho da Carteira de CRIs
    ROUND(
      (nanvl(nota_pvp, 0.0) * 0.40) + 
      (nanvl(nota_dy, 0.0) * 0.40) + 
      (nanvl(nota_portfolio, 0.0) * 0.20), 2
    ) AS score_final,
    -- Papel possui segmento fixo unificado na Gold e Staging
    'Papel' AS segmento_alvo
  FROM scores_calculados
"""

df_scores = spark.sql(qry_scoring)
df_scores.createOrReplaceTempView("v_scores_papel_reais_calculados")

# %%
# 3. Geração do Ranking Geral e carga com INSERT OVERWRITE
qry_insert_ranking_papel = f"""
  INSERT OVERWRITE {catalogo}.{schema}.stg_scoring_papel
  SELECT 
    ticker,
    data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    6.5 AS taxa_media_ipca, 
    1.8 AS taxa_media_cdi, 
    valor_portfolio_cri AS max_concentracao_devedor, 
    score_final,
    -- REGRA DE DESEMPATE FÍSICA: Se empatar no score_final, prioriza fundo com menor dívida e mais cotistas
    ROW_NUMBER() OVER (
      PARTITION BY segmento_alvo 
      ORDER BY score_final DESC, alavancagem_percentual ASC, total_cotistas DESC
    ) AS posicao_ranking,
    CURRENT_TIMESTAMP() AS data_calculo,
    segmento_alvo,
    -- Gravação dos campos de desempate físicos na Staging
    total_cotistas,
    alavancagem_percentual,
    caixa_percentual
  FROM v_scores_papel_reais_calculados
"""

print(f"Gravando classificação e ranking REAL de Papel em: {catalogo}.{schema}.stg_scoring_papel...")
spark.sql(qry_insert_ranking_papel)
print("✅ Scoring REAL, Dividend Yield Isento e Ranking Desempatado de Papel concluídos com SUCESSO!")